In [3]:
import time
start = time.perf_counter()

import pandas as pd
import requests
import time

df = pd.read_parquet("../data/fuel_prices/silver/fuels_prices")

df = df.dropna(how='all')

In [4]:
df

,nm_region,nm_state,nm_city,nm_gas_station,nm_neighborhood,nm_fuel_type,dt_date,nu_fuel_price,nm_unit_of_measurement,nm_fuel_brand,dt_year,dt_month,ab_state,uf_city
0,Norte,Acre,Rio Branco,Auto Posto Amapa - Eireli,Areal,Diesel,2022-01-03,6.09,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
1,Norte,Acre,Rio Branco,Auto Posto Amapa - Eireli,Areal,Diesel S10,2022-01-03,6.12,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
2,Norte,Acre,Rio Branco,Auto Posto Acauan Ltda,Vila Acre,Diesel,2022-01-03,6.09,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
3,Norte,Acre,Rio Branco,Auto Posto Acauan Ltda,Vila Acre,Diesel S10,2022-01-03,6.12,R$ / litro,Vibra Energia,2022,1,AC,ac_rio_branco
4,Norte,Acre,Rio Branco,Auto Posto Correntao Ltda,Santa Ines,Diesel,2022-01-03,6.08,R$ / litro,Branca,2022,1,AC,ac_rio_branco
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3197862,Sudeste,Minas Gerais,Barbacena,Posto Das Flores Ltda,Pontilhao,Gasolina Aditivada,2026-02-28,6.59,R$ / litro,Raizen,2026,2,MG,mg_barbacena
3197863,Sudeste,Minas Gerais,Barbacena,Posto Das Flores Ltda,Pontilhao,Etanol,2026-02-28,4.59,R$ / litro,Raizen,2026,2,MG,mg_barbacena
3197864,Sudeste,Minas Gerais,Barbacena,Posto Sete De Setembro Ltda,Centro,Gasolina,2026-02-28,6.19,R$ / litro,Raizen,2026,2,MG,mg_barbacena
3197865,Sudeste,Minas Gerais,Barbacena,Posto Sete De Setembro Ltda,Centro,Gasolina Aditivada,2026-02-28,6.49,R$ / litro,Raizen,2026,2,MG,mg_barbacena


In [3]:
import datetime
import findspark
import pandas as pd
findspark.init()

import sys
print(sys.version)

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

import requests

spark = (
    SparkSession.builder
    .appName("Fuel Prices in Brazil")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

df = spark.read.parquet("databases/fuel_prices/silver/fuels_prices")

df = df.dropna(how='all')

df = df.withColumn('dt_date_month_start', to_date(concat_ws('-', col('dt_year'), col('dt_month'), lit('01'))))

df = (
    df
    .groupBy('nm_region', 'nm_state', 'nm_city', 'nm_fuel_type', 'dt_date_month_start', 'nm_fuel_brand', 'uf_city')
    .agg(
        mean('nu_fuel_price').alias('avg_fuel_price')
    )
    .withColumn("avg_fuel_price_r", round(col("avg_fuel_price"), 2))
    .orderBy('dt_date_month_start', 'nm_state', 'nm_city', 'nm_fuel_type')
)


3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [5]:
most_recent_date = df.select('dt_date_month_start').distinct().orderBy(col('dt_date_month_start').desc()).first()['dt_date_month_start'].strftime('%Y-%m-%d')

rows = df.select(
    'dt_date_month_start',
    'avg_fuel_price'
).collect()

list_of_dates = [row.dt_date_month_start for row in rows]
list_of_values = [row.avg_fuel_price for row in rows]

dates_as_strings = [
    d.isoformat() if isinstance(d, (datetime.date, datetime.datetime)) else d 
    for d in list_of_dates
]

payload = {
    'dates': dates_as_strings,
    'values': list_of_values,
    'currency': 'BRL',
    'present_date': most_recent_date
}

headers = {
    "Content-Type": "application/json"
}

values_inflation_adjusted = requests.post('https://financial-utilities-api.onrender.com/inflation_adjustment', json=payload, headers=headers)

if values_inflation_adjusted.status_code != 200:
    values_inflation_adjusted = requests.post('https://financial-utilities-api.onrender.com/inflation_adjustment', json=payload, headers=headers)

df_inflation_adjusted_values = spark.createDataFrame(pd.DataFrame(values_inflation_adjusted.json()))

df_inflation_adjusted_values = df_inflation_adjusted_values.toDF('date', 'original_value', 'inflation_adjusted_avg_fuel_price', 'inflation_perc')

df_inflation_adjusted_values = (
    df_inflation_adjusted_values
    .withColumn("original_value", round(col("original_value"), 2))
)

df = (
    df
    .withColumn("avg_fuel_price", round(col("avg_fuel_price"), 2))
)


Py4JJavaError: An error occurred while calling o188.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 17.0 failed 1 times, most recent failure: Lost task 0.0 in stage 17.0 (TID 23) (JoaoPaiva executor driver): java.io.FileNotFoundException: C:\Users\joaov\AppData\Local\Temp\blockmgr-d76eb9cc-3b11-4324-8bd6-3f7fd6730781\15\temp_shuffle_e49b868c-3d99-42a7-81f0-b510c1083b68 (O sistema não pode encontrar o caminho especificado)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.B

In [ ]:
df.limit(5).show()

Py4JJavaError: An error occurred while calling o168.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 16.0 failed 1 times, most recent failure: Lost task 0.0 in stage 16.0 (TID 22) (192.168.2.3 executor driver): java.io.FileNotFoundException: C:\Users\joaov\AppData\Local\Temp\blockmgr-23ead533-a632-4f98-9092-b52d3132e828\09\temp_shuffle_3b1e28ec-fa04-4412-84d6-aeb792122a9b (O sistema não pode encontrar o caminho especificado)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.Bypa

In [ ]:
rows_inf_adj = df.select(
    'inflation_adjusted_avg_fuel_price'
).collect()

list_inf_adj_values = [row.inflation_adjusted_avg_fuel_price for row in rows_inf_adj]

df.withColumn('inflation_adjusted_avg_fuel_price', list_inf_adj_values).show()

{"ts": "2025-12-31 17:54:21.226", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `inflation_adjusted_avg_fuel_price` cannot be resolved. Did you mean one of the following? [`avg_fuel_price`, `avg_fuel_price_r`, `dt_date_month_start`, `nm_fuel_brand`, `nm_fuel_type`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor8.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o143.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `inflation_adjusted_avg_fuel_price` cannot be resolved. Did you mean one of the following? [`avg_fuel_price`, `avg_fuel_price_r`, `dt_date_month_start`, `nm_fuel_brand`, `nm_fuel_type`]. SQLSTATE: 42703;\n'P

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `inflation_adjusted_avg_fuel_price` cannot be resolved. Did you mean one of the following? [`avg_fuel_price`, `avg_fuel_price_r`, `dt_date_month_start`, `nm_fuel_brand`, `nm_fuel_type`]. SQLSTATE: 42703;
'Project ['inflation_adjusted_avg_fuel_price]
+- Project [nm_region#0, nm_state#1, nm_city#2, nm_fuel_type#5, dt_date_month_start#15, nm_fuel_brand#9, uf_city#13, round(avg_fuel_price#16, 2) AS avg_fuel_price#55, avg_fuel_price_r#33]
   +- Sort [dt_date_month_start#15 ASC NULLS FIRST, nm_state#1 ASC NULLS FIRST, nm_city#2 ASC NULLS FIRST, nm_fuel_type#5 ASC NULLS FIRST], true
      +- Project [nm_region#0, nm_state#1, nm_city#2, nm_fuel_type#5, dt_date_month_start#15, nm_fuel_brand#9, uf_city#13, avg_fuel_price#16, round(avg_fuel_price#16, 2) AS avg_fuel_price_r#33]
         +- Aggregate [nm_region#0, nm_state#1, nm_city#2, nm_fuel_type#5, dt_date_month_start#15, nm_fuel_brand#9, 

In [ ]:
df = (
    df
    .join(
        df_inflation_adjusted_values.select(
            'date',
            'original_value',
            'inflation_adjusted_avg_fuel_price'
        ),
        (df.dt_date_month_start == df_inflation_adjusted_values.date) &
        (df.avg_fuel_price == df_inflation_adjusted_values.original_value),
        how='left'
    )
)

df = df.drop('original_value', 'avg_fuel_price_r', 'date')

Py4JJavaError: An error occurred while calling o165.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 15.0 failed 1 times, most recent failure: Lost task 0.0 in stage 15.0 (TID 19) (192.168.2.3 executor driver): java.io.FileNotFoundException: C:\Users\joaov\AppData\Local\Temp\blockmgr-23ead533-a632-4f98-9092-b52d3132e828\25\temp_shuffle_9bad8e8c-3867-4fe5-a1c4-d112325e41fa (O sistema não pode encontrar o caminho especificado)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.Bypa

In [ ]:
df.write.mode("overwrite").format("parquet").save("databases/fuel_prices/gold/fuels_prices")

# Inflation Adjustment Tests

In [24]:
import time
start = time.perf_counter()

import pandas as pd
import requests
import time

df = pd.read_parquet("../data/fuels_prices/silver/fuels_prices")

df = df.dropna(how='all')

df['dt_date_month_start'] = pd.to_datetime(
    df['dt_year'].astype(str) + '-' + df['dt_month'].astype(str) + '-01'
)

df = df.groupby(
    ['nm_region', 'nm_state', 'nm_city', 'nm_fuel_type', 'dt_date_month_start', 'nm_fuel_brand', 'uf_city'],
    as_index=False
)['nu_fuel_price'].mean()

df = df.rename(columns={'nu_fuel_price': 'avg_fuel_price'})
df['avg_fuel_price'] = df['avg_fuel_price'].round(2)

df = df.sort_values(['dt_date_month_start', 'nm_state', 'nm_city', 'nm_fuel_type'])

most_recent_date = df['dt_date_month_start'].max().strftime('%Y-%m-%d')

df['dt_date_month_start'] = df['dt_date_month_start'].astype(str)

In [10]:
most_recent_date

'2026-03-01'

In [25]:
df

,nm_region,nm_state,nm_city,nm_fuel_type,dt_date_month_start,nm_fuel_brand,uf_city,avg_fuel_price
82788,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Atem' S,ac_cruzeiro_do_sul,6.67
82789,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Branca,ac_cruzeiro_do_sul,6.70
82790,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Ipiranga,ac_cruzeiro_do_sul,6.63
82791,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Vibra Energia,ac_cruzeiro_do_sul,6.68
82922,Norte,Acre,Cruzeiro Do Sul,Diesel S10,2022-01-01,Atem' S,ac_cruzeiro_do_sul,6.68
...,...,...,...,...,...,...,...,...
103726,Norte,Tocantins,Porto Nacional,Etanol,2026-03-01,Raizen,to_porto_nacional,5.79
103826,Norte,Tocantins,Porto Nacional,Gasolina,2026-03-01,Branca,to_porto_nacional,6.80
103827,Norte,Tocantins,Porto Nacional,Gasolina,2026-03-01,Raizen,to_porto_nacional,7.06
103927,Norte,Tocantins,Porto Nacional,Gasolina Aditivada,2026-03-01,Branca,to_porto_nacional,6.83


In [21]:

# Inflation Adjustment
df_monthly_ipca = pd.read_csv('../data/inflation_adjustment/bronze/monthly_inflation_index.csv')
df_monthly_ipca = df_monthly_ipca[['Date', 'CPI Value']]

In [22]:
df_monthly_ipca

,Date,CPI Value
0,1994-07-01,915.93
1,1994-08-01,932.97
2,1994-09-01,947.24
3,1994-10-01,972.06
4,1994-11-01,999.37
...,...,...
376,2025-11-01,7378.94
377,2025-12-01,7403.29
378,2026-01-01,7427.72
379,2026-02-01,7479.71


In [18]:
present_value_cpi = df_monthly_ipca[df_monthly_ipca['Date'] == most_recent_date]['CPI Value'].values[0]

In [27]:
df = df.merge(df_monthly_ipca, left_on='dt_date_month_start', right_on='Date', how='left')

In [28]:
df['inflation_adjustment_factor'] = present_value_cpi / df['CPI Value']

In [32]:
df['inflation_adjusted_avg_fuel_price'] = round(df['avg_fuel_price'] * df['inflation_adjustment_factor'], 2)

In [34]:
df.drop(columns=['Date', 'CPI Value', 'inflation_adjustment_factor'], inplace=True)

In [35]:
df

,nm_region,nm_state,nm_city,nm_fuel_type,dt_date_month_start,nm_fuel_brand,uf_city,avg_fuel_price,inflation_adjusted_avg_fuel_price
0,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Atem' S,ac_cruzeiro_do_sul,6.67,8.18
1,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Branca,ac_cruzeiro_do_sul,6.70,8.22
2,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Ipiranga,ac_cruzeiro_do_sul,6.63,8.13
3,Norte,Acre,Cruzeiro Do Sul,Diesel,2022-01-01,Vibra Energia,ac_cruzeiro_do_sul,6.68,8.19
4,Norte,Acre,Cruzeiro Do Sul,Diesel S10,2022-01-01,Atem' S,ac_cruzeiro_do_sul,6.68,8.19
...,...,...,...,...,...,...,...,...,...
311135,Norte,Tocantins,Porto Nacional,Etanol,2026-03-01,Raizen,to_porto_nacional,5.79,5.79
311136,Norte,Tocantins,Porto Nacional,Gasolina,2026-03-01,Branca,to_porto_nacional,6.80,6.80
311137,Norte,Tocantins,Porto Nacional,Gasolina,2026-03-01,Raizen,to_porto_nacional,7.06,7.06
311138,Norte,Tocantins,Porto Nacional,Gasolina Aditivada,2026-03-01,Branca,to_porto_nacional,6.83,6.83
